# Example 03: Auto-Simulate on Signal

When a signal fires above threshold, automatically run a mimic-world simulation
on the affected companies and get decision recommendations within 60 seconds.

In [ ]:
# Requires: pip install mimic mimic-world mimic-signal

from mimic import Twin
from mimic_world import World, Scenario
from mimic_signal import SignalMonitor, Signal

In [ ]:
# Build the world
world = World()
world.add_twin(Twin.from_ticker("WMT"))   # heavy Asia supply exposure
world.add_twin(Twin.from_ticker("FDX"))   # logistics
world.add_twin(Twin.from_ticker("AAPL"))  # semiconductor supply chain

# Create monitor with world context
monitor = SignalMonitor(world=world, threshold=0.6)
monitor.watch(["gdelt", "sec_8k", "fred", "newsapi"])

In [ ]:
@monitor.on_signal(threshold=0.65)
def handle_signal_with_simulation(signal: Signal, affected_twins: list):
    print(f"\nSIGNAL: {signal.title}")
    print(f"Severity: {signal.severity:.2f} | Confidence: {signal.confidence:.2f}")
    print(f"Affected: {[t.context.ticker for t in affected_twins]}")

    if not affected_twins:
        print("No affected twins — skipping simulation")
        return

    # Auto-simulate
    try:
        scenario = Scenario.from_signal(signal)
        affected_world = World.subset(world, affected_twins)
        result = affected_world.run(scenario)

        print("\nSIMULATION RESULTS:")
        print(result.financial_impacts)
        print(f"\nTop recommendation: {result.recommendations[0]}")
    except Exception as e:
        print(f"Simulation failed: {e}")

In [ ]:
import asyncio

task = monitor.start_async()

# Run for 10 minutes
await asyncio.sleep(600)

monitor.stop()

print(f"\nTotal signals processed: {len(monitor.signal_log())}")